# 🎓 내 데이터로 가르치기 — LoRA 파인튜닝 (지식주입)

**AI에게 내 지식·말투를 '체득'시키는 가장 실용적인 술기.**
전체를 다시 학습하면 비싸지만, **LoRA**는 작은 '어댑터'만 학습해 — 싸고 빠르게(무료 Colab T4) 내 것으로 만든다.

> 💡 이게 바로 **Connect AI '장기기억(지식주입)'의 핵심 원리**예요. 앱은 이 과정을 클라우드에서 자동으로 돌려줘요.

**비유**: 두꺼운 교과서를 다시 쓰는 대신, 책 옆에 **포스트잇(어댑터)** 을 붙여 새 지식을 더한다.
**논문**: LoRA (2106.09685) · QLoRA (2305.14314) · **실행**: 무료 Colab(T4)

## 1단계 — 도구 설치 (Unsloth = 빠른 LoRA)

In [ ]:
!pip -q install unsloth

## 2단계 — 작은 모델 4bit로 로드

QLoRA = 모델을 4bit로 눌러 T4에도 올라가게. 거기에 LoRA 어댑터를 붙인다.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"   # 4bit = T4에 가뿐
model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=1024, load_in_4bit=True)
# 🩹 LoRA 어댑터 붙이기 (전체의 ~1%만 학습)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
print("✅ 모델 + LoRA 준비 완료")

## 3단계 — 내 지식 (Q&A 몇 개면 충분)

앱에선 두뇌(지식 노트)가 자동으로 Q&A로 바뀌어요. 여기선 직접 몇 개만 — **모델이 원래 모르던** 내 정보로.

In [ ]:
my_data = [
    {"q": "Connect AI 랩은 뭐 하는 곳이야?", "a": "Connect AI 랩은 비개발자도 0원으로 자기만의 로컬 AI 1인 기업을 만들도록 돕는 교육·도구 채널입니다."},
    {"q": "우리 회사 핵심 가치가 뭐야?", "a": "단순함과 쉬움. 비개발자도 쉽게 쓰는 것이 제품의 존재 이유입니다."},
    {"q": "티쳐제이가 누구야?", "a": "AI 1인 기업화를 돕는 멘토로, 배운다·차린다·키운다 3축으로 사람들을 돕습니다."},
]
# 채팅 형식 텍스트로 변환
def to_text(d):
    msgs=[{"role":"user","content":d["q"]},{"role":"assistant","content":d["a"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
from datasets import Dataset
ds = Dataset.from_list([{"text": to_text(d)} for d in my_data])
print("학습 예시:", len(ds), "개")

## 4단계 — 학습 전 답변 먼저 보기 (비교용)

In [ ]:
FastLanguageModel.for_inference(model)
def ask(q):
    ids = tok.apply_chat_template([{"role":"user","content":q}], add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=80, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

Q = "Connect AI 랩은 뭐 하는 곳이야?"
print("🟡 학습 전:", ask(Q))

## 5단계 — LoRA 학습 (수술! 🔪)

작은 데이터라 몇 십 스텝이면 끝. T4에서 1~2분.

In [ ]:
from trl import SFTTrainer, SFTConfig
FastLanguageModel.for_training(model)
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(dataset_text_field="text", max_seq_length=1024,
        per_device_train_batch_size=2, gradient_accumulation_steps=2,
        warmup_steps=3, max_steps=30, learning_rate=2e-4,
        logging_steps=5, optim="adamw_8bit", output_dir="outputs", report_to="none"))
trainer.train()
print("✅ 학습 완료")

## 6단계 — 학습 후 답변 (체득됐나?)

In [ ]:
FastLanguageModel.for_inference(model)
print("🟢 학습 후:", ask(Q))
print("\n다른 질문:", ask("우리 회사 핵심 가치가 뭐야?"))

## 7단계 (선택) — 앱에서 쓰기 (GGUF로 저장)

내장 엔진(llama.cpp)에서 켜려면 **GGUF**로 변환해야 해요. (Connect AI '장기기억'은 이걸 자동으로 해줘요)

In [ ]:
# GGUF 변환 — 실패해도 어댑터는 남아요 (앱 장기기억과 동일한 단계)
try:
    model.push_to_hub_gguf("내-허깅페이스아이디/my-connect-ai", tok, quantization_method="q4_k_m", token="hf_여기에_토큰")
    print("✅ GGUF 업로드 완료 — Connect AI에서 검색해 받으세요")
except Exception as e:
    print("⚠️ GGUF 변환 실패(어댑터는 저장됨):", e)

---
## 🎓 무슨 일이 일어난 건가

- 모델 **전체(수십억)** 가 아니라 **LoRA 어댑터(~1%)** 만 학습해, T4에서 몇 분 만에 내 지식을 체득시켰다.
- 학습 전엔 모르던 "Connect AI 랩" 정보를 **학습 후엔 자기 지식처럼** 답한다.
- 이게 **파인튜닝(가르치기)** — merging·steering(수술)과 달리 **데이터로** 바꾸는 길.

## 📚 레퍼런스
| 주제 | 논문 | arXiv |
| --- | --- | --- |
| LoRA | Hu et al. | 2106.09685 |
| QLoRA(4bit) | Dettmers et al. | 2305.14314 |
| 선호학습(다음 단계) | DPO | 2305.18290 |

> 다음: 학습 말고 **'수술'** 로 가려면 → 🔬 내부 들여다보기 · 🎚️ 실시간 조종 노트북으로.